### Prepare dialogue 

In [1]:
import pandas as pd

with open("../../own_script/dialogue_3/dialogue_3.txt", "r", encoding="utf-8") as f:
    lines = [l.strip() for l in f if l.strip()]

df = pd.DataFrame({
    "utterance_id": range(1, len(lines)+1),
    "text": lines,
})
df.head()

,utterance_id,text
0,1,I've been feeling really overwhelmed lately. M...
1,2,"When I feel the palpitations, my heart starts ..."
2,3,"Well, the last time I went to the doctor, they..."
3,4,I guess if I could really believe that my hear...
4,5,"I think trying those techniques could help, bu..."


### Load text/VAD model

In [2]:
import torch, transformers
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)


torch: 2.5.1+cu121
transformers: 4.57.0


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "RobroKools/vad-bert"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()


tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/864 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

### Predict function

In [ ]:
import numpy as np

def predict_vad(texts):
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc)
    # logits shape: [batch, 3] = [V, A, D]
    vad = out.logits.cpu().numpy()
    return vad  # np.array [batch,3]


### Check value range of vac-bert

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tok = AutoTokenizer.from_pretrained("RobroKools/vad-bert")
model = AutoModelForSequenceClassification.from_pretrained("RobroKools/vad-bert")

samples = [
    "I feel terrible and hopeless.",
    "I feel completely neutral.",
    "I feel amazing and so happy!",
]

for s in samples:
    inputs = tok(s, return_tensors="pt")
    with torch.no_grad():
        out = model(**inputs).logits.squeeze().tolist()
    print(s, "-> VAD:", out)


In [ ]:
samples = [
    "I want to die. I hate everything.",
    "I feel completely empty and numb.",
    "This is fine.",
    "I'm a little annoyed.",
    "I'm so excited I can't stop screaming!",
    "I feel calm, peaceful, and relaxed.",
]

vals = []
for s in samples:
    inputs = tok(s, return_tensors="pt")
    with torch.no_grad():
        out = model(**inputs).logits.squeeze().tolist()  # [V,A,D]
    print(s, "->", out)
    vals.append(out)

import numpy as np
vals = np.array(vals)
print("Valence range:", vals[:,0].min(), vals[:,0].max())
print("Arousal range:", vals[:,1].min(), vals[:,1].max())
print("Dominance range:", vals[:,2].min(), vals[:,2].max())


### Test on dialogue 2 and save into dataframe

In [ ]:
vad = predict_vad(df["text"].tolist())
df["valence_text"] = vad[:, 0]
df["arousal_text"] = vad[:, 1]
df["dominance_text"] = vad[:, 2]

df.head()

### Check with own dataset

In [ ]:
import numpy as np

v = df["valence_text"].to_numpy()
a = df["arousal_text"].to_numpy()

print("Valence min/max:", v.min(), v.max())
print("Valence q1/median/q3:", np.quantile(v, [0.25, 0.5, 0.75]))

print("Arousal min/max:", a.min(), a.max())
print("Arousal q1/median/q3:", np.quantile(a, [0.25, 0.5, 0.75]))


### Scale mismatch issue --> Normalization

- The vac-bert I used has scale mismatch from wagner ([-1, 1]) where as vac-bert scale is ([1, 5])

- To solve this, I implement the normalization method to vac-bert scale to be compatible with wagner and the paper

In [ ]:
import numpy as np

# กำหนดช่วง text สมมติเป็น 1..5
V_MIN, V_MAX = 1.0, 5.0

def to_minus1_1(x, xmin=V_MIN, xmax=V_MAX):
    return 2 * (x - xmin) / (xmax - xmin) - 1  # map [xmin,xmax] -> [-1,1]

df["valence_text_n"]  = to_minus1_1(df["valence_text"])
df["arousal_text_n"]  = to_minus1_1(df["arousal_text"])
df["dominance_text_n"] = to_minus1_1(df["dominance_text"])


In [ ]:
df

### Assert mutual absolute scale --> Check by revert back to its previous form 

- The vac-bert I used has scale mismatch from wagner ([-1, 1]) where as vac-bert scale is ([1, 5])

- To check this, I implement the revert method to vac-bert scale and assert it to contain the same value for original scale then there is unchanged in absolute meaning in value

- The output must be 0

In [ ]:
# map 1..5 -> -1..1
def one5_to_minus1_1(x, xmin=1.0, xmax=5.0):
    x01 = (x - xmin) / (xmax - xmin)
    return 2*x01 - 1

def minus1_1_to_one5(y, xmin=1.0, xmax=5.0):
    x01 = (y + 1) / 2
    return x01 * (xmax - xmin) + xmin

diff = df["arousal_text"] - minus1_1_to_one5(df["arousal_text_n"])
print(diff.abs().max())   # ควร ~ 0 (มีแค่ numerical noise ระดับ 1e-7)


### Check rank & order value

- The output must be 1

In [ ]:
# index ของค่า arousal สูงสุด ก่อนและหลัง normalize ต้องเป็นอันเดียวกัน
orig_argmax = df["arousal_text"].idxmax()
norm_argmax = df["arousal_text_n"].idxmax()
print(orig_argmax, norm_argmax)

# หรือ correlation ระหว่างค่าเดิมกับค่าที่ normalize ควร = 1
df[["arousal_text", "arousal_text_n"]].corr()


### Check distribution

In [ ]:
print(df["arousal_text"].describe())
print(df["arousal_text_n"].describe())


### Save to .csv format

In [ ]:
df.to_csv("../../own_script/dialogue_2/dialogue_2_vad_text.csv", index=False)